> 在线讲义：[第 7 章 · FAST：先压缩，再变词](https://wikieden.github.io/vla-from-scratch/lessons/07-fast.html) · 本书《VLA 从零到 π0》

# 第 7 章 · FAST：先压缩，再变词

配套讲义：仓库里 `vla-deep-dive/notes/lessons/07-fast.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `copy_last_accuracy(seq)` | 6 / 9 |
| 第 2 步 | `dct`、`idct`、`keep_low(x, k)` | 匀速只有 1 个非零系数；加减速留 3 个误差 0.00014；来回抖留 6 个仍误差 0.011 |
| 第 2 步 | `fast_tokens(x, γ)` | γ = 10 时 50 个系数只剩 2 个非零 |

DCT 用正交的 DCT-II，自己用 NumPy 写，不依赖 scipy。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `copy_last_accuracy`**：一个只会"抄上一个"的模型，从第 2 个开始猜，正确率多少。

In [ ]:
def copy_last_accuracy(seq):
    """一个只会"抄上一个"的模型，从第 2 个开始猜，正确率多少。"""
    hits = sum(a == b for a, b in zip(seq, seq[1:]))
    return hits, len(seq) - 1

**函数 `dct_matrix`**：正交 DCT-II 矩阵：第 k 行是第 k 个频率的余弦。

In [ ]:
def dct_matrix(n):
    """正交 DCT-II 矩阵：第 k 行是第 k 个频率的余弦。"""
    k = np.arange(n)[:, None]
    i = np.arange(n)[None, :]
    m = np.cos(np.pi * (i + 0.5) * k / n) * np.sqrt(2 / n)
    m[0] /= np.sqrt(2)
    return m

**函数 `dct`**：一段动作 → 频率系数（低频在前）。

In [ ]:
def dct(x):
    """一段动作 → 频率系数（低频在前）。"""
    x = np.asarray(x, float)
    return dct_matrix(len(x)) @ x

**函数 `idct`**：频率系数 → 一段动作（正交矩阵的逆就是转置）。

In [ ]:
def idct(c):
    """频率系数 → 一段动作（正交矩阵的逆就是转置）。"""
    c = np.asarray(c, float)
    return dct_matrix(len(c)).T @ c

**函数 `keep_low`**：只保留前 k 个低频系数再还原，返回重建结果和最大误差。

In [ ]:
def keep_low(x, k):
    """只保留前 k 个低频系数再还原，返回重建结果和最大误差。"""
    c = dct(x)
    c[k:] = 0
    r = idct(c)
    return r, float(np.abs(r - x).max())

**函数 `fast_tokens`**：FAST 的第 ③ 步：系数乘 γ 再取整，数一数还剩几个非零（越少越好压）。

In [ ]:
def fast_tokens(x, gamma):
    """FAST 的第 ③ 步：系数乘 γ 再取整，数一数还剩几个非零（越少越好压）。"""
    q = np.round(dct(x) * gamma).astype(int)
    return q, int(np.count_nonzero(q))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 相邻的词太像")
seq = [128, 128, 129, 129, 129, 130, 130, 130, 131, 131]
hits, n = copy_last_accuracy(seq)
print(f"  {seq}：抄上一个猜对 {hits} / {n}")

In [ ]:
print("\n第二步 DCT：先说形状，再补细节")
flat = np.full(50, 0.01)
c = dct(flat)
print(f"  匀速 50 步：非零系数 {np.count_nonzero(np.round(c, 12))} 个，第 0 个 = {c[0]:.4f}")
t = np.linspace(0, 1, 50)
smooth = 0.01 * np.sin(np.pi * t) ** 2  # 加速 → 匀速 → 减速（编的）
for k in (1, 3, 6):
    _, err = keep_low(smooth, k)
    print(f"  加减速：保留 {k} 个低频，最大误差 {err:.5f}")
jitter = 0.01 * (-1.0) ** np.arange(50)  # 来回抖
_, err = keep_low(jitter, 6)
print(f"  来回抖：保留 6 个低频，最大误差 {err:.4f}（能量全在高频）")
_, nz = fast_tokens(smooth / 0.01, gamma=10)
print(f"  归一化后乘 γ = 10 取整：50 个系数里非零 {nz} 个")

## 3. 代码里才看得到的

1. `dct_matrix` 第 0 行要再除以 √2，才是正交矩阵；少了这一步，`idct(dct(x))` 就回不到 x。
2. 匀速那一段的 DC 系数是 0.01 × √50 ≈ 0.0707，不是 0.01：正交 DCT 会把"平均值"放大 √N 倍。

## 4. 自己改一改

- 把 γ 从 10 改成 100，看非零系数变多少。
- 自己写一段"先匀速、中间突然停一下"的动作，看要保留几个低频。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 相邻太像，抄上一个就高分，什么都没学会；
> 先说形状再补细节，DCT 压一压，BPE 并一并，字少还有料。